# Slocum L0 exploration — first pass

Quick, **L0-only** look at a mission, before any processing decisions are
made. No L1/L2 needed — just the raw decoded archive
(`slocum-process-mission <N> --from-ogdb --steps l0`).

Answers the three things you need before setting `processing.l1_time_range`
(and `unused_sensors`) in `deployment.yml` — see
`docs/user-guide/processing-a-mission.md` step 3:

1. **Track** — where did the glider actually go (`m_lat`/`m_lon`)?
2. **Depth vs time** — when did real diving start/stop
   (`m_present_time` vs `m_depth`, the flight side — present from the very
   first file, unlike the science stream)?
3. **Science channels** — which `sci_*` sensors actually logged data, over
   what range, and for how much of the deployment?

For everything past this — L1/L2, T-S, gridded sections, interactive views
— use `data_exploration.ipynb`.


## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pyglider.utils import nmea2deg   # Slocum ddmm.mmmm -> decimal degrees

plt.rcParams["figure.figsize"] = (14, 5)
pd.set_option("display.max_rows", None)      # never truncate the science-channel table
pd.set_option("display.max_colwidth", None)


## 1. Load L0

In [ ]:
MISSION = 2                 # mission number, or a python/missions/ dir prefix ("002")

DATA_ROOT = Path("/Data/gfi/projects/slocum/data/delayed")

_token = f"{int(MISSION):03d}" if str(MISSION).isdigit() else str(MISSION)
_matches = [d for d in sorted(DATA_ROOT.glob(f"{_token}*")) if d.is_dir()]
assert len(_matches) == 1, f"expected one folder for {_token!r}, found {[d.name for d in _matches]}"
deployment_dir = _matches[0]
name = deployment_dir.name

l0_file = deployment_dir / "pyglider" / "L0" / f"{name}_L0.nc"
assert l0_file.is_file(), (
    f"no L0 file at {l0_file} -- run "
    f"`slocum-process-mission {MISSION} --from-ogdb --steps l0` first"
)

l0 = xr.open_dataset(l0_file)
t_all = pd.to_datetime(l0["time"].values)
print(name)
print(f"L0: {dict(l0.sizes)} samples, {t_all.min()} -> {t_all.max()}  "
      f"({t_all.max() - t_all.min()})")


## 2. Track (`m_lat` / `m_lon`)

In [ ]:
# Slocum m_lat/m_lon are raw NMEA ddmm.mmmm, NOT decimal degrees --
# e.g. 6015.234 = 60 deg 15.234' = 60.2539 deg. Convert before plotting;
# pyglider's own L1 build does the same conversion internally for its
# latitude/longitude, but L0 keeps the raw sensor value untouched.
lat = nmea2deg(l0["m_lat"].values)
lon = nmea2deg(l0["m_lon"].values)
t = pd.to_datetime(l0["time"].values)
ok = np.isfinite(lat) & np.isfinite(lon)

fig, ax = plt.subplots(figsize=(8, 8))
sc = ax.scatter(lon[ok], lat[ok], c=t[ok].astype("int64"), s=6, cmap="viridis")
ax.plot(lon[ok], lat[ok], "-", lw=0.4, alpha=0.4, color="grey")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.set_aspect("equal", "datalim")
ax.set_title(f"{name} -- L0 track ({int(ok.sum())} fixes)")
cb = fig.colorbar(sc, ax=ax, shrink=0.7)
cb.ax.set_yticklabels([pd.Timestamp(v).strftime("%m-%d") for v in cb.get_ticks()])


## 3. Depth vs time — suggest a start/stop window

In [ ]:
# m_present_time is the flight computer's own clock, logged as raw Unix
# epoch seconds (not auto-decoded like the L0 `time` coordinate) -- convert
# with unit="s". Using the raw sensor pair (not L0's merged `time` axis)
# is deliberate: it's exactly what m_depth was actually logged against.
t_depth = pd.to_datetime(l0["m_present_time"].values, unit="s")
z = l0["m_depth"].values
ok = np.isfinite(t_depth) & np.isfinite(z)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(t_depth[ok], z[ok], ".", ms=2)
ax.invert_yaxis()
ax.set_xlabel("m_present_time"); ax.set_ylabel("m_depth [m]")
ax.set_title(f"{name} -- L0 depth vs time (flight clock)")
ax.grid(alpha=0.3)


In [ ]:
# Heuristic first guess for processing.l1_time_range: a day only counts as
# "diving" if it has enough samples AND real depth range -- filters out
# bench tests / a surface checkout where m_depth barely moves. Diving days
# are then grouped into episodes (allowing small gaps -- a glider can sit
# at the surface a day or two mid-deployment for comms/GPS); the longest
# episode is the suggestion, others are reported separately so an isolated
# pre-deployment trial (see mission 002) doesn't get bridged into it.
#
# This is a depth-only heuristic, not a verdict on science-data quality --
# always cross-check against the plot above and the science-channel plots
# below. (Mission 002's FLNTU is the cautionary tale the other way: OGDB
# assigning a sensor to a mission doesn't mean it was switched on -- the
# data is what decides, not the config.)
MIN_SAMPLES_PER_DAY = 100
MIN_DEPTH_RANGE_M = 5.0
MAX_GAP_DAYS = 2   # merge diving days into one episode across gaps this short

s = pd.Series(z[ok], index=t_depth[ok]).sort_index()
daily = s.resample("1D").agg(["count", "min", "max"])
daily.columns = ["n", "zmin", "zmax"]
daily["range"] = daily["zmax"] - daily["zmin"]
diving = daily[(daily["n"] >= MIN_SAMPLES_PER_DAY) & (daily["range"] >= MIN_DEPTH_RANGE_M)]

print(daily.round(1))

if diving.empty:
    print("\nno day meets the diving threshold -- lower MIN_SAMPLES_PER_DAY / "
          "MIN_DEPTH_RANGE_M and re-run")
else:
    gap = diving.index.to_series().diff() > pd.Timedelta(days=MAX_GAP_DAYS)
    episode = gap.cumsum()
    episodes = [grp.index for _, grp in diving.groupby(episode)]
    episodes.sort(key=len, reverse=True)

    for ep in episodes:
        tag = "longest -- suggested" if ep is episodes[0] else "separate episode"
        print(f"\n{tag}: {ep.min().date()} .. {ep.max().date()} ({len(ep)} day(s))")

    start = episodes[0].min().date()
    end = (episodes[0].max() + pd.Timedelta(days=1)).date()   # +1 so the last day isn't clipped
    print(f"\nsuggested processing.l1_time_range: ['{start}', '{end}']")


## 4. Science channels

In [ ]:
# Every sci_* variable that carried data in L0 -- build_l0 already drops
# any sensor that was declared in sensors.txt but never logged a finite
# value, so everything listed here logged *something*. "Some data" isn't
# the same as "data for the whole deployment window" -- check n / first /
# last against the deployment span above, and the plots below.
sci_vars = sorted(v for v in l0.data_vars
                  if v.startswith("sci_") and v != "sci_m_present_time")

rows = []
for v in sci_vars:
    x = l0[v].values
    ok_v = np.isfinite(x)
    tv = t_all[ok_v]
    rows.append({"variable": v, "n": int(ok_v.sum()),
                 "first": tv.min(), "last": tv.max(),
                 "min": np.nanmin(x), "max": np.nanmax(x)})
summary = pd.DataFrame(rows).set_index("variable")
summary


In [ ]:
ncols = 2
nrows = -(-len(sci_vars) // ncols)   # ceil
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3 * nrows), squeeze=False)
for ax, v in zip(axes.flat, sci_vars):
    x = l0[v].values
    ok_v = np.isfinite(x)
    ax.plot(t_all[ok_v], x[ok_v], ".", ms=2)
    ax.set_title(f"{v}  (n={int(ok_v.sum())})   [{l0[v].attrs.get('units', '')}]")
    ax.grid(alpha=0.3)
for ax in axes.flat[len(sci_vars):]:
    ax.axis("off")
fig.suptitle(f"{name} -- all science channels present in L0", y=1.02)
fig.tight_layout()


## Next

- Real diving window narrower/wider than suggested above? Set
  `processing.l1_time_range` in `deployment.yml` accordingly.
- A science channel flat/empty for (part of) the window despite OGDB
  assigning that sensor? Add it to `processing.unused_sensors` (see the
  runbook) rather than leaving a dead calibration claim in the metadata.
- Then: `slocum-process-mission <N> --from-ogdb --regenerate` to run L1/L2,
  and switch to `data_exploration.ipynb` to look at the result.
